# Загварчлал ба харьцуулалт

Цэвэр өгөгдөл дээр 4 регрессийн загварыг сургаж, тэдгээрийн нарийвчлалыг харьцуулна.

**Ажиллуулах загварууд:**
1. Шугаман регресс (baseline)
2. Lasso регресс (L1 регулярчилал)
3. Шийдвэрийн мод (шугаман бус, тайлбарлахад хялбар)
4. Random Forest (ансамбль)

**Зорилтот хувьсагч**: `log_price` (EDA-аас log-transform хийх нь зүйтэй гэж тогтоосон).

**Хэмжүүр**: RMSE, MAE, R²

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style='whitegrid', font_scale=1.05)
RANDOM_STATE = 42

df = pd.read_csv('../data/listings_clean.csv')
df.shape

## 1. Хувьсагч сонголт

**Чухал**: `price_per_m2` нь үнээс задарсан тул feature болгож болохгүй (data leakage). `log_price`-ыг target болгож, `price_mnt`-ийг хасна.

In [ ]:
TARGET = 'log_price'

NUMERIC_FEATURES = [
    'area_m2', 'rooms', 'floor', 'total_floors', 'floor_ratio',
    'building_age', 'has_balcony', 'balcony_count', 'has_garage',
    'has_elevator', 'is_finished',
]
CATEGORICAL_FEATURES = [
    'district_grouped', 'window_type', 'door_type', 'floor_material',
]

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
y = df[TARGET].copy()

print(f'X хэлбэр: {X.shape}')
print(f'y хэлбэр: {y.shape}')
print(f'\nДутуу утгууд (X):\n{X.isna().sum()[X.isna().sum() > 0]}')

## 2. Сургалт/тестийн хуваалт

80/20 хуваалт, `random_state=42` нь дахин ажиллуулахад үр дүн ижил байх баталгаа.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f'Сургалт: {X_train.shape[0]} зар')
print(f'Тест:    {X_test.shape[0]} зар')

## 3. Preprocessing pipeline

- **Тоо**: дутууг median-аар нөхөх → standard scale (шугаман загварт чухал)
- **Категори**: дутууг 'unknown'-аар нөхөх → one-hot encoding

Pipeline дотор хийж buulah нь cross-validation-д data leakage гарахаас сэргийлнэ.

In [ ]:
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, NUMERIC_FEATURES),
    ('cat', categorical_pipe, CATEGORICAL_FEATURES),
])

## 4. Загварууд тодорхойлох

In [ ]:
models = {
    'Шугаман регресс': LinearRegression(),
    'Lasso (α=0.01)': Lasso(alpha=0.01, random_state=RANDOM_STATE, max_iter=10000),
    'Ridge (α=1.0)': Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'Шийдвэрийн мод': DecisionTreeRegressor(max_depth=8, min_samples_leaf=5, random_state=RANDOM_STATE),
    'Random Forest': RandomForestRegressor(
        n_estimators=300, max_depth=None, min_samples_leaf=3,
        n_jobs=-1, random_state=RANDOM_STATE
    ),
}

list(models.keys())

## 5. Cross-validation харьцуулалт

5-fold CV-р RMSE-ийг тооцно. Хэмжүүр нь `log_price` дээр учир `exp` хийгээд бодит ₮-руу буулгадаг түвшинд RMSE өөрөөр харагдана — энэ хэсэгт зөвхөн загвар хоорондын харьцуулалт.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = []

for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    rmse_scores = -cross_val_score(
        pipe, X_train, y_train, cv=cv,
        scoring='neg_root_mean_squared_error', n_jobs=-1
    )
    r2_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='r2', n_jobs=-1)
    cv_results.append({
        'Загвар': name,
        'CV RMSE (log)': rmse_scores.mean(),
        'CV RMSE std': rmse_scores.std(),
        'CV R²': r2_scores.mean(),
    })

cv_df = pd.DataFrame(cv_results).sort_values('CV RMSE (log)')
cv_df.round(4)

## 6. Тест өгөгдөл дээр эцсийн үнэлгээ

In [ ]:
test_results = []
predictions = {}
fitted_pipes = {}

for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    
    y_pred_log = pipe.predict(X_test)
    
    # Лог-ээс бодит ₮-руу буулгана
    y_pred_mnt = np.exp(y_pred_log)
    y_test_mnt = np.exp(y_test)
    
    rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
    rmse_mnt = np.sqrt(mean_squared_error(y_test_mnt, y_pred_mnt))
    mae_mnt = mean_absolute_error(y_test_mnt, y_pred_mnt)
    r2 = r2_score(y_test, y_pred_log)
    
    test_results.append({
        'Загвар': name,
        'RMSE (log)': rmse_log,
        'RMSE (сая ₮)': rmse_mnt / 1e6,
        'MAE (сая ₮)': mae_mnt / 1e6,
        'R²': r2,
    })
    predictions[name] = y_pred_mnt
    fitted_pipes[name] = pipe

test_df = pd.DataFrame(test_results).sort_values('RMSE (log)')
test_df.round(3)

## 7. Таамаг vs бодит үнэ

In [ ]:
y_test_mnt = np.exp(y_test)
fig, axes = plt.subplots(1, len(models), figsize=(4 * len(models), 4), sharey=True)

for ax, (name, y_pred_mnt) in zip(axes, predictions.items()):
    ax.scatter(y_test_mnt / 1e6, y_pred_mnt / 1e6, alpha=0.5, s=20)
    lim = max(y_test_mnt.max(), y_pred_mnt.max()) / 1e6
    ax.plot([0, lim], [0, lim], 'r--', linewidth=1, label='y = x')
    ax.set_xlabel('Бодит үнэ (сая ₮)')
    ax.set_title(name, fontsize=10)
    ax.legend(fontsize=8)

axes[0].set_ylabel('Таамагласан үнэ (сая ₮)')
plt.suptitle('Тестийн өгөгдөл дээр таамаг vs бодит үнэ')
plt.tight_layout(); plt.show()

## 8. Шугаман регрессийн коэффициентүүд

Эерэг коэффициент → үнийг өсгөнө, сөрөг → бууруулна.  
Зорилтот нь `log_price` тул коэффициент нь "тухайн хувьсагч 1 нэгж нэмэгдэхэд үнэ ойролцоогоор exp(coef)-1 хувиар өөрчлөгдөнө" гэсэн утгатай.

In [ ]:
lr_pipe = fitted_pipes['Шугаман регресс']
feature_names = lr_pipe.named_steps['prep'].get_feature_names_out()
coefs = lr_pipe.named_steps['model'].coef_

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coef': coefs,
    'pct_effect': (np.exp(coefs) - 1) * 100,  # ойролцоогоор % өөрчлөлт
}).sort_values('coef', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 8))
top = coef_df.head(15)
colors = ['steelblue' if c > 0 else 'coral' for c in top['coef']]
ax.barh(top['feature'], top['coef'], color=colors)
ax.invert_yaxis()
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Коэффициент (log_price дээр)')
ax.set_title('Шугаман регрессийн ТОП-15 хүчин зүйл')
plt.tight_layout(); plt.show()

coef_df.head(10).round(3)

## 9. Random Forest — feature importance

In [ ]:
rf_pipe = fitted_pipes['Random Forest']
feature_names = rf_pipe.named_steps['prep'].get_feature_names_out()
importances = rf_pipe.named_steps['model'].feature_importances_

imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances,
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(8, 8))
top = imp_df.head(15)
ax.barh(top['feature'], top['importance'], color='forestgreen')
ax.invert_yaxis()
ax.set_xlabel('Чухлын зэрэг (Gini importance)')
ax.set_title('Random Forest — ТОП-15 хүчин зүйл')
plt.tight_layout(); plt.show()

imp_df.head(10).round(4)

## 10. Hyperparameter tuning (Random Forest)

GridSearchCV-р хамгийн сайн загвар олно. Энэ алхам урт ажиллана (~1-3 минут).

In [ ]:
rf_grid = {
    'model__n_estimators': [200, 400],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_leaf': [1, 3, 5],
}
rf_pipe = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestRegressor(n_jobs=-1, random_state=RANDOM_STATE)),
])
search = GridSearchCV(
    rf_pipe, rf_grid, cv=cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1, verbose=0,
)
search.fit(X_train, y_train)

print(f'Шилдэг параметрүүд: {search.best_params_}')
print(f'CV RMSE (log): {-search.best_score_:.4f}')

best_rf = search.best_estimator_
y_pred_log = best_rf.predict(X_test)
y_pred_mnt = np.exp(y_pred_log)
rmse_mnt = np.sqrt(mean_squared_error(np.exp(y_test), y_pred_mnt)) / 1e6
r2 = r2_score(y_test, y_pred_log)
print(f'Тест RMSE: {rmse_mnt:.1f} сая ₮')
print(f'Тест R²: {r2:.3f}')

## 11. Дүгнэлт

_(Бүрэн өгөгдөл дээр ажиллуулсны дараа дараах байдлаар бөглөх):_

1. **Хамгийн сайн загвар**: _<нэр>_ — тестийн RMSE _<X сая ₮>_, R² _<Y>_
2. **Үнэд хамгийн нөлөөтэй хүчин зүйл**:
   - Талбай (м²) — _коэффициент / importance_
   - Дүүрэг (Хан-Уул, Сүхбаатар үнэтэй)
   - Барилгын нас (сөрөг хамаарал)
3. **Шугаман vs мод**: Random Forest нь шугаман регрессээс RMSE-ийг _Z%_-аар бууруулсан → шугаман бус хамаарал бий
4. **Хязгаарлалт**:
   - Зөвхөн нийслэлийн зар (хөдөө орохгүй)
   - Зарын үнэ ≠ зарагдсан үнэ (бид arrival үнийг хардаг)
   - Цаг хугацааны нөлөөг тооцоогүй
5. **Цаашид сайжруулах**:
   - GPS координат, тойрон оршин буй сургууль/эмнэлэг зэрэг feature
   - Gradient boosting (XGBoost) туршилт
   - Хугацааны цуваагаар зах зээлийн тренд оруулах